In [ ]:
# ===============================================================
# 🚦 Forecasting of Smart City Traffic Patterns
# Author: Kaif Shakil Qureshi
# ===============================================================

# Step 1: Install Required Libraries
!pip install pmdarima tensorflow scikit-learn matplotlib pandas numpy

# Step 2: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from math import sqrt
from pmdarima import auto_arima
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from google.colab import files
import glob
import os

# Step 3: Upload Files
print("📂 Please upload your train and test CSV files...")
uploaded = files.upload()

# Step 4: Detect the Latest Uploaded Files
train_file = sorted(glob.glob("train_aWnotuB*.csv"))[-1]
test_file = sorted(glob.glob("datasets_8494_11879_test_BdBKkAj*.csv"))[-1]

print(f"✅ Using files:\nTrain ➜ {train_file}\nTest ➜ {test_file}")

# Step 5: Load Datasets
train = pd.read_csv(train_file)
test = pd.read_csv(test_file)

print("\n📊 Dataset Shapes:")
print("Train:", train.shape)
print("Test:", test.shape)
print("\n📋 Columns in Train:", list(train.columns))

# Step 6: Detect the Traffic Volume Column Automatically
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) == 0:
    raise ValueError("No numeric columns found. Please check your dataset!")
traffic_col = num_cols[0]
print(f"\n✅ Detected traffic volume column: '{traffic_col}'")

# Step 7: Clean Data
train = train.fillna(method='ffill')
test = test.fillna(method='ffill')

# Step 8: Visualize Traffic Data
plt.figure(figsize=(10,4))
plt.plot(train[traffic_col], label='Traffic Volume')
plt.title("Traffic Volume Over Time")
plt.xlabel("Time")
plt.ylabel("Volume")
plt.legend()
plt.show()

# Step 9: Train ARIMA Model
arima_model = auto_arima(train[traffic_col], seasonal=False, stepwise=True, trace=False)
print("\n✅ ARIMA Model Fitted Successfully")

# Step 10: Forecast with ARIMA
n_periods = len(test)
arima_pred = arima_model.predict(n_periods=n_periods)

plt.figure(figsize=(10,4))
plt.plot(train[traffic_col], label='Train')
plt.plot(range(len(train), len(train)+len(test)), arima_pred, label='ARIMA Forecast', color='red')
plt.title("ARIMA Traffic Forecast")
plt.legend()
plt.show()

# Step 11: Prepare Data for LSTM
series = train[traffic_col].values
n_input = 10
n_features = 1
generator = TimeseriesGenerator(series, series, length=n_input, batch_size=32)

# Step 12: Build and Train LSTM Model
model = Sequential([
    LSTM(64, activation='relu', input_shape=(n_input, n_features)),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
print("\n🧠 Training LSTM model...")
model.fit(generator, epochs=20, verbose=1)
print("✅ LSTM Training Complete")

# Step 13: Forecast Using LSTM
pred_list = []
batch = series[-n_input:].reshape((1, n_input, n_features))

for i in range(len(test)):
    pred = model.predict(batch, verbose=0)[0]
    pred_list.append(pred)
    batch = np.append(batch[:,1:,:], [[pred]], axis=1)

lstm_pred = np.array(pred_list)

plt.figure(figsize=(10,4))
plt.plot(train[traffic_col], label='Train')
plt.plot(range(len(train), len(train)+len(test)), lstm_pred, label='LSTM Forecast', color='orange')
plt.title("LSTM Traffic Forecast")
plt.legend()
plt.show()

# Step 14: Evaluate Models (if test data has actual traffic column)
if traffic_col in test.columns:
    y_true = test[traffic_col].values
    mae_arima = mean_absolute_error(y_true, arima_pred)
    rmse_arima = sqrt(mean_squared_error(y_true, arima_pred))
    mae_lstm = mean_absolute_error(y_true, lstm_pred)
    rmse_lstm = sqrt(mean_squared_error(y_true, lstm_pred))

    print("\n📈 MODEL PERFORMANCE")
    print(f"ARIMA ➜ MAE: {mae_arima:.2f} | RMSE: {rmse_arima:.2f}")
    print(f"LSTM  ➜ MAE: {mae_lstm:.2f} | RMSE: {rmse_lstm:.2f}")
else:
    print("\n⚠️ No actual traffic column found in test dataset for evaluation.")

# Step 15: Save Forecasts
output = pd.DataFrame({
    'ARIMA_Predicted': arima_pred,
    'LSTM_Predicted': lstm_pred.flatten()
})
output.to_csv('Traffic_Forecast_Output.csv', index=False)
print("\n✅ Forecast saved as 'Traffic_Forecast_Output.csv'")

# Step 16: Display Preview
print("\n🔍 Forecast Output Preview:")
print(output.head())

# Step 17: Option to Download
from google.colab import files
files.download('Traffic_Forecast_Output.csv')
